In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datetime import datetime
import random
from functools import partial
from typing import List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import numpy as np
from datasets import Dataset, disable_progress_bar
import wandb

from src.data_processing import DataPreprocessing
from src.tokenizer import train_tokenizer
from src.gpt import preprocess_datasets
import warnings

warnings.filterwarnings("ignore",)
disable_progress_bar()



/Users/nottreepat/Downloads/KilterTransformer-2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialize wandb
wandb.init(
    project="climb-gpt-shuffle",
    name=f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config={
        "architecture": "GPT-SetLoss",
        "n_embd": 256,
        "n_head": 4,
        "n_layer": 6,
        "n_positions": 128,
        "dropout": 0.1,
        "epochs": 30,
        "batch_size": 16,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "gradient_accumulation_steps": 1,
        "early_stopping_patience": 5,
        "allow_empty_prompt": True,
        "min_prefix_len": 1,  # Changed from 3 to allow BOS-only
    }
)

run_name = wandb.run.name

# Switch between local and Colab paths
# OUT_DIR = f"/content/drive/MyDrive/KilterTransformer/models/climb_gpt/{run_name}"
OUT_DIR = f"models/climb_gpt/{run_name}"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

wandb: Currently logged in as: treepatchantaurai (treepatchantaurai-me) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cpu


In [3]:
# Load and split data
dp = DataPreprocessing()
datasets = dp.load_climbs()

# 80-10-10 split
train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_test['train'],
    'val': val_test['train'],
    'test': val_test['test']
}

print(f"Train size: {len(datasets['train'])}")
print(f"Val size: {len(datasets['val'])}")
print(f"Test size: {len(datasets['test'])}")

Loaded 76992 routes from cache data/climbs_cleaned.csv
Train size: 61593
Val size: 7699
Test size: 7700


In [4]:
# Train tokenizer
tokenizer = train_tokenizer(datasets, OUT_DIR)
wandb.config.update({"vocab_size": tokenizer.vocab_size})

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.special_tokens_map}")

# Tokenize datasets
datasets = preprocess_datasets(datasets, tokenizer)
print("✓ Datasets tokenized")

Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('finish1180', 459), ('feet1147', 324), ('start1524', 1629), ('hand1107', 166), ('finish1571', 1819), ('hand1521', 1618), ('start1252', 745), ('start1384', 1273), ('start1172', 425), ('finish1376', 1243)]

Sample encodings:

Input: angle35_grade14_feet1595_start1400
Tokens: ['[BOS]', 'angle35', 'grade14', 'feet1595', '[UNK]', '[EOS]', ('[PAD]', 19)]

Input: angle40_grade15_feet1595_start1596_hand1597_finish1598
Tokens: ['[BOS]', 'angle40', 'grade15', 'feet1595', 'start1596', 'hand1597', 'finish1598', '[EOS]', ('[PAD]', 17)]
Saving tokenizer to models/climb_gpt/run_20251102_195640
Vocabulary size: 1932
Special tokens: {'bos_token': '[BOS]', 'eos_token': '[EOS]', 'unk_token': '[UNK]', 'pad_token': '[PAD]'}
✓ Datasets tokenized


In [5]:
def create_shuffle_datasets(dataset, tokenizer, min_prefix_len=1, augment=True):
    """
    Create training examples with diverse prefix lengths.
    
    If augment=True (training set):
        - Original dataset is preserved
        - Adds augmented examples:
          * 5% Empty prompt: [BOS] only
          * 5% Angle only: [BOS, angle]
          * 15% Angle + Grade: [BOS, angle, grade]
          * 75% Random splits: 2 different random positions
    
    If augment=False (val/test set):
        - Returns original dataset unchanged
    
    Args:
        dataset: HuggingFace dataset with 'input_ids'
        tokenizer: trained tokenizer
        min_prefix_len: minimum prefix length (1 = just BOS)
        augment: whether to augment (True for train, False for val/test)
    """
    if not augment:
        # For validation/test: return original dataset as-is
        # Convert to list format for consistency
        original_examples = []
        for example in dataset:
            tokens = example['input_ids']
            actual_tokens = [t for t in tokens if t != tokenizer.pad_token_id]
            
            if len(actual_tokens) < 5:
                continue
            
            eos_idx = len(actual_tokens) - 1
            
            # Single random split for evaluation
            split_point = random.randint(max(3, min_prefix_len), eos_idx - 1)
            original_examples.append({
                'input_ids': actual_tokens[:split_point],
                'remaining': actual_tokens[split_point:],
            })
        
        return Dataset.from_list(original_examples)
    
    
    
    # First, add original full sequences as random split examples
    all_examples = []
    for example in dataset:
        tokens = example['input_ids']
        actual_tokens = [t for t in tokens if t != tokenizer.pad_token_id]
        
        if len(actual_tokens) < 5:
            continue
        
        eos_idx = len(actual_tokens) - 1
        
        # Original: one random split
        split_point = random.randint(max(3, min_prefix_len), eos_idx - 1)
        all_examples.append({
            'input_ids': actual_tokens[:split_point],
            'remaining': actual_tokens[split_point:],
        })
    
    # Now add augmented examples
    augmented_examples = []
    for example in dataset:
        tokens = example['input_ids']
        actual_tokens = [t for t in tokens if t != tokenizer.pad_token_id]
        
        if len(actual_tokens) < 5:
            continue
        
        eos_idx = len(actual_tokens) - 1
        
        # Random assignment to augmentation type
        rand = random.random()
        
        if rand < 0.05:
            # 5% Empty prompt: [BOS] -> predict everything
            augmented_examples.append({
                'input_ids': [actual_tokens[0]],
                'remaining': actual_tokens[1:],
            })
        
        elif rand < 0.10:  # FIX: was 0.05, should be 0.10 (5% + 5%)
            # 5% Angle only: [BOS, angle] -> predict grade + holds + EOS
            augmented_examples.append({
                'input_ids': actual_tokens[:2],
                'remaining': actual_tokens[2:],
            })
        
        elif rand < 0.25:  # FIX: 0.10 + 0.15 = 0.25
            # 15% Angle + Grade: [BOS, angle, grade] -> predict holds + EOS
            augmented_examples.append({
                'input_ids': actual_tokens[:3],
                'remaining': actual_tokens[3:],
            })
        
        else:
            # 75% Random splits: add TWO different random positions
            for _ in range(2):
                split_point = random.randint(max(3, min_prefix_len), eos_idx - 2)
                
                augmented_examples.append({
                    'input_ids': actual_tokens[:split_point],
                    'remaining': actual_tokens[split_point:],
                })
    
    # Combine original + augmented
    all_examples.extend(augmented_examples)
    
    return Dataset.from_list(all_examples)


train_dataset = create_shuffle_datasets(datasets['train'], tokenizer, augment=True)
val_dataset = create_shuffle_datasets(datasets['val'], tokenizer, augment=False)

print(f"Training examples: {len(train_dataset):,}")
print(f"  Original: {len(datasets['train']):,}")
print(f"  Augmented: {len(train_dataset) - len(datasets['train']):,}")
print(f"  Augmentation factor: {len(train_dataset) / len(datasets['train']):.2f}x")
print(f"Validation examples: {len(val_dataset):,}")

# Log dataset sizes
wandb.config.update({
    "train_size": len(train_dataset),
    "train_original": len(datasets['train']),
    "train_augmented": len(train_dataset) - len(datasets['train']),
    "val_size": len(val_dataset),
    "augmentation_factor": len(train_dataset) / len(datasets['train']),
})

# Inspect examples
print("\n=== Example Training Samples ===")
for i in np.random.randint(0, len(train_dataset), size=3):
    sample = train_dataset[i]
    print(f"\nExample {i+1}:")
    print(f"  Prefix length: {len(sample['input_ids'])}")
    print(f"  Prefix: {tokenizer.decode(sample['input_ids'])}")
    print(f"  Remaining count: {len(sample['remaining'])}")
    print(f"  Remaining: {[tokenizer.decode([t]) for t in sample['remaining'][:5]]}...")  # Show first 5

Training examples: 169,489
  Original: 61,593
  Augmented: 107,896
  Augmentation factor: 2.75x
Validation examples: 7,699

=== Example Training Samples ===

Example 26526:
  Prefix length: 7
  Prefix: [BOS] angle55 grade16 feet1142 start1163 feet1192 start1195
  Remaining count: 10
  Remaining: ['feet1243', 'hand1247', 'feet1260', 'hand1279', 'hand1281']...

Example 16984:
  Prefix length: 8
  Prefix: [BOS] angle25 grade21 start1136 feet1182 start1199 hand1222 hand1274
  Remaining count: 6
  Remaining: ['hand1310', 'hand1339', 'finish1389', 'feet1480', 'feet1543']...

Example 82461:
  Prefix length: 14
  Prefix: [BOS] angle30 grade17 start1152 start1171 hand1204 hand1256 hand1270 hand1322 hand1353 finish1390 finish1391 feet1452 feet1455
  Remaining count: 4
  Remaining: ['feet1471', 'feet1525', 'feet1567', '[EOS]']...


In [6]:
def shuffle_collate_fn(batch: List[Dict], tokenizer):
    """
    Collate batch for set loss training.
    
    Args:
        batch: List of dicts with 'input_ids' and 'remaining'
        tokenizer: for pad_token_id
    
    Returns:
        dict with padded input_ids, attention_mask, and remaining (as list)
    """
    # Extract sequences
    input_ids_list = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    remaining_list = [torch.tensor(x["remaining"], dtype=torch.long) for x in batch]
    
    # Pad input_ids
    input_ids = nn.utils.rnn.pad_sequence(
        input_ids_list, 
        batch_first=True, 
        padding_value=tokenizer.pad_token_id
    )
    
    # Create attention mask (1 for real tokens, 0 for padding)
    attention_mask = nn.utils.rnn.pad_sequence(
        [torch.ones(len(ids), dtype=torch.long) for ids in input_ids_list],
        batch_first=True,
        padding_value=0
    )
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "remaining": remaining_list,  # Keep as list of tensors
    }
    

In [7]:
from transformers.modeling_outputs import CausalLMOutputWithCrossAttentions

class KilterShuffleGPT(nn.Module):
    """
    GPT model that predicts over sets of valid next tokens.
    Loss: -log(sum of probabilities over all valid next tokens)
    
    This allows the model to learn that multiple tokens are valid at each step,
    making it suitable for unordered set prediction (climbing route generation).
    """
    
    def __init__(
        self, 
        vocab_size: int,
        n_embd: int = 256,
        n_head: int = 4,
        n_layer: int = 6,
        n_positions: int = 128,
        dropout: float = 0.1
        ):
        super().__init__()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            n_positions=n_positions,
            n_ctx=n_positions,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)
        self.config = config
        self.vocab_size = vocab_size

    def forward(self, input_ids, attention_mask=None, remaining=None, **kwargs):
        """
        Args:
            input_ids: [batch_size, seq_len] - prefix tokens
            attention_mask: [batch_size, seq_len]
            remaining: List[Tensor] - list of valid next token IDs per example
                       remaining[i] = tensor([token_id1, token_id2, ...])
        """
        # Forward pass through GPT
        outputs = self.model(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            labels=None  # We compute custom loss
        )
        
        # # Get logits for the last position (next token prediction)
        logits = outputs.logits  # [batch_size, seq_len, vocab_size]
        last_logits = logits[:, -1, :]  # [batch_size, vocab_size]
        
        loss = None
        if remaining is not None:
            loss = self.compute_set_loss(last_logits, remaining)

        return CausalLMOutputWithCrossAttentions(
            loss=loss,
            logits=logits,
            past_key_values=None,
            hidden_states=None,
            attentions=None,
        )
    
    def compute_set_loss(self, logits, remaining):
        """
        Compute log-sum-exp loss over valid token sets.
        
        Loss = -log(sum_over_valid_tokens(P(token)))
             = -logsumexp(log_probs[valid_tokens])
        
        This encourages the model to assign high total probability mass
        to all valid next tokens, not just a single token.
        
        Args:
            logits: [batch_size, vocab_size]
            remaining: List[Tensor] of valid token IDs per example
        """
        batch_size = logits.size(0)
        losses = []
        
        # Compute log probabilities once
        log_probs = F.log_softmax(logits, dim=-1)  # [batch_size, vocab_size]
        
        for i in range(batch_size):
            valid_tokens = remaining[i]  # Tensor of valid token IDs
            
            if len(valid_tokens) == 0:
                continue
            
            # Get log probs for valid tokens
            valid_log_probs = log_probs[i, valid_tokens]  # [num_valid]
            
            # Log-sum-exp: log(sum(exp(log_probs)))
            # This gives us: log(P(token1) + P(token2) + ... + P(tokenN))
            # We want to maximize this, so minimize its negative
            loss_i = -torch.logsumexp(valid_log_probs, dim=0)
            losses.append(loss_i)
        
        if len(losses) == 0:
            return torch.tensor(0.0, device=logits.device)
        
        # Average over batch
        return torch.stack(losses).mean()


# Initialize model
model = KilterShuffleGPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=256,
    n_head=4,
    n_layer=6,
    n_positions=128,
    dropout=0.1
)

print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

Model initialized with 5,266,432 parameters


In [ ]:
# Watch with wandb
wandb.watch(model.model, log="all", log_freq=1000)

# Data collator with tokenizer bound
collate_fn = partial(shuffle_collate_fn, tokenizer=tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=100,
    eval_steps=30, #############
    save_steps=30, #############
    num_train_epochs=1, #############
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    report_to="wandb",
    remove_unused_columns=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    run_name=run_name,
    dataloader_pin_memory=False,
    )

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=val_dataset, #######
    tokenizer=tokenizer,  
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )

# print("\n=== Starting Training ===")
# trainer.train()

# # Save
# model.model.save_pretrained(OUT_DIR)
# tokenizer.save_pretrained(OUT_DIR)
# print(f"\n✓ Model saved to {OUT_DIR}")

# wandb.finish()
# print("\n✓ Training complete!")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.



=== Starting Training ===


Step,Training Loss,Validation Loss
30,No log,No log


KeyError: "The `metric_for_best_model` training argument is set to 'eval_loss', which is not found in the evaluation metrics. The available evaluation metrics are: []. Consider changing the `metric_for_best_model` via the TrainingArguments."